# Gold: lab_test_trends

Builds `<catalog>.gold.lab_test_trends` (one row per test type per month) from `silver.lab_results` per the STTM `sttm_silver_to_gold.xlsx`, with calendar month/quarter names and a LAG-based month-over-month abnormal rate trend. Full rebuild via CREATE OR REPLACE TABLE AS SELECT.

In [0]:
import re

dbutils.widgets.text("catalog", "databrickscaresync")
catalog = dbutils.widgets.get("catalog").strip()

# Guard against injection through the catalog widget
if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: '{catalog}'")

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.lab_test_trends"
print(f"[SOURCE] {source_schema}  ->  [TARGET] {target_table}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

select_sql = f"""
WITH monthly AS (
  -- Driving table: lab_results grouped by test type and month of test_date
  SELECT
    test_name,
    year(test_date)                      AS year,
    month(test_date)                     AS month,
    CAST(date_trunc('month', test_date) AS DATE) AS month_start,
    COUNT(*)                             AS total_tests,
    COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical') AS critical_test_count
  FROM {source_schema}.lab_results
  GROUP BY test_name, year(test_date), month(test_date), CAST(date_trunc('month', test_date) AS DATE)
),
with_rates AS (
  SELECT
    m.*,
    CAST(m.abnormal_test_count / m.total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(m.critical_test_count / m.total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM with_rates r
)
SELECT
  CAST(w.test_name AS STRING)                AS test_name,
  CAST(w.year AS INT)                        AS year,
  CAST(w.month AS INT)                       AS month,
  CAST(c.month_name AS STRING)               AS month_name,
  -- silver.calendar has no quarter_name column; derive it from calendar.quarter (e.g. 'Q1')
  CAST(concat('Q', c.quarter) AS STRING)     AS quarter_name,
  CAST(w.total_tests AS INT)                 AS total_tests,
  CAST(w.abnormal_test_count AS INT)         AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)         AS critical_test_count,
  w.abnormal_rate_pct                        AS abnormal_rate_pct,
  w.critical_rate_pct                        AS critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct,
  CASE
    WHEN w.prev_month_abnormal_rate_pct IS NULL THEN NULL
    ELSE CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2))
  END                                        AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN {source_schema}.calendar c
  ON c.date_key = CAST(date_format(w.month_start, 'yyyyMMdd') AS INT)
"""

spark.sql(f"CREATE OR REPLACE TABLE {target_table} AS {select_sql}")
print(f"[DONE] Rebuilt {target_table}")

In [0]:
check = spark.sql(f"""
SELECT
  (SELECT COUNT(*) FROM {source_schema}.lab_results)       AS source_tests,
  SUM(total_tests)                                         AS gold_total_tests,
  COUNT(*)                                                 AS gold_rows,
  COUNT(DISTINCT test_name, year, month)                   AS gold_distinct_test_months
FROM {target_table}
""").first()
print(check)

# Grain: one row per test type per month; every lab test counted exactly once
assert check.gold_rows == check.gold_distinct_test_months, "Duplicate test_name/month rows in gold"
assert check.gold_total_tests == check.source_tests, "Sum of total_tests differs from silver.lab_results row count"

display(spark.table(target_table).orderBy("test_name", "year", "month").limit(10))